# Stations

## Imports

In [1]:
import numpy as np
import pandas as pd
import pprint as pp
import folium
import json
import os
import re

# https://www.datacamp.com/tutorial/settingwithcopywarning-pandas
# https://saturncloud.io/blog/how-to-disable-warnings-in-jupyter-notebook/
import warnings

# Not good, but the warnings were annoying me!!
# warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore')

In [2]:
from generic_code.ContaminantManagerJSON import ContaminantManagerJSON
from generic_code.StationEDAHelper import StationEDAHelper

## Dataset information

* Air quality measure stations of the city of Barcelona.
* As of 04/04/2019 the dataset structure changes.
    * All data of 2019 uses the new standard ?? It seems so

* [Stations Dataset](https://opendata-ajuntament.barcelona.cat/data/en/dataset/qualitat-aire-estacions-bcn)

## Get contaminants metadata

In [3]:
contaminants_json_file_path: str = './../data/gold/contaminants/contaminants.json'
contaminant_manager = ContaminantManagerJSON(contaminants_json_file_path)

In [4]:
pp.pprint(contaminant_manager.get_all_contaminants())

{'CO': {'codes': [6, 106], 'unit': 'mg/m³'},
 'NO': {'codes': [7, 107], 'unit': 'µg/m³'},
 'NO2': {'codes': [8, 108], 'unit': 'µg/m³'},
 'NOx': {'codes': [12, 112], 'unit': 'µg/m³'},
 'O3': {'codes': [14, 114], 'unit': 'µg/m³'},
 'PM10': {'codes': [10, 110], 'unit': 'µg/m³'},
 'PM2_5': {'codes': [9, 109], 'unit': 'µg/m³'},
 'SO2': {'codes': [1, 101], 'unit': 'µg/m³'}}


## List dataset files

In [5]:
bronze_stations_folder_path = './../data/bronze/stations'

StationEDAHelper.print_all_station_files(bronze_stations_folder_path) 


Found 6 files:
 * '2018_stations.csv'
 * '2019_stations.csv'
 * '2021_stations.csv'
 * '2022_stations.csv'
 * '2023_stations.csv'
 * '2025_stations.csv'


In [6]:
# keys = {complete_years, existent_years, missing_station_years, previous_year_mapping}
years_metadata =  StationEDAHelper.calculate_station_years_metadata(bronze_stations_folder_path)

Station files range from year 2018 to 2025.
Existent years: [2018, 2019, 2021, 2022, 2023, 2025]
Missing station files for years: {2024, 2020}
Complete years should be: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


### Study column names variation

In [7]:
all_station_columns = StationEDAHelper.fetch_all_station_columns(
    bronze_stations_folder_path,
    years_metadata['existent_years'])

pp.pprint(all_station_columns)

{2018: ['nom_cabina',
        'codi_dtes',
        'zqa',
        'codi_eoi',
        'longitud',
        'latitud',
        'Ubicacio',
        'Codi_Districte',
        'Nom_Districte',
        'Codi_Barri',
        'Nom_Barri',
        'Ocupacio_sol',
        'Emissions_Properes',
        'Contaminant_1',
        'Contaminant_2',
        'Contaminant_3'],
 2019: ['Estacio',
        'nom_cabina',
        'codi_dtes',
        'zqa',
        'codi_eoi',
        'Longitud',
        'Latitud',
        'ubicacio',
        'Codi_districte',
        'Nom_districte',
        'Codi_barri',
        'Nom_barri',
        'Clas_1',
        'Clas_2',
        'Codi_Contaminant'],
 2021: ['Estacio',
        'nom_cabina',
        'codi_dtes',
        'zqa',
        'codi_eoi',
        'Longitud',
        'Latitud',
        'ubicacio',
        'Codi_districte',
        'Nom_districte',
        'Codi_barri',
        'Nom_barri',
        'Clas_1',
        'Clas_2',
        'Codi_Contaminant'],
 2022: ['

In [64]:
# create a function to do this 
# needs to call other functions
# make this a method of a class
print("*** Current vs Previous Year Columns Comparison: ***\n")
for year in existent_years[1:]:
    previous_year = previous_year_mapping[year]
    print(f"Current year {year} vs Previous Year {previous_year}:")
    print(f"* Current year columns: {all_station_columns[year]}")
    print(f"* Previous year columns: {all_station_columns[previous_year]}")
    print(f"* Difference with previous year: ")
    diff = set(all_station_columns[year]) - set(all_station_columns[previous_year])
    print(f"    * {diff}") # improve - show new columns and removed columns
    print('\n', '-' * 100, end='\n\n')

*** Current vs Previous Year Columns Comparison: ***

Current year 2019 vs Previous Year 2018:
* Current year columns: ['Estacio', 'nom_cabina', 'codi_dtes', 'zqa', 'codi_eoi', 'Longitud', 'Latitud', 'ubicacio', 'Codi_districte', 'Nom_districte', 'Codi_barri', 'Nom_barri', 'Clas_1', 'Clas_2', 'Codi_Contaminant']
* Previous year columns: ['nom_cabina', 'codi_dtes', 'zqa', 'codi_eoi', 'longitud', 'latitud', 'Ubicacio', 'Codi_Districte', 'Nom_Districte', 'Codi_Barri', 'Nom_Barri', 'Ocupacio_sol', 'Emissions_Properes', 'Contaminant_1', 'Contaminant_2', 'Contaminant_3']
* Difference with previous year: 
    * {'Clas_2', 'Codi_barri', 'Nom_barri', 'ubicacio', 'Latitud', 'Nom_districte', 'Codi_Contaminant', 'Estacio', 'Longitud', 'Clas_1', 'Codi_districte'}

 ----------------------------------------------------------------------------------------------------

Current year 2021 vs Previous Year 2019:
* Current year columns: ['Estacio', 'nom_cabina', 'codi_dtes', 'zqa', 'codi_eoi', 'Longitud', 

In [69]:
columns_from_2019_foward_es = ['Estacio', 'nom_cabina', 'codi_dtes',
    'zqa', 'codi_eoi', 'Longitud', 'Latitud', 'ubicacio', 'Codi_districte',
    'Nom_districte', 'Codi_barri', 'Nom_barri', 'Clas_1', 'Clas_2', 'Codi_Contaminant']

columns_from_2019_foward_en = ['station_number', 'station_name', 'station_code', 
    'aqzc', 'eoi_code', 'longitude', 'latitude', 'location', 'district_code',
    'district_name', 'neighborhood_code', 'neighborhood_name', 'class_1', 'class_2', 'contaminant_code']

columns_from_2019_foward_en_description = [
    'Station number', # naybe can be ignored (each station has multiple rows, one per contaminant measured)
    'Name of the station',
    'Station code', # station identifier (should be unique) -> lookup key
    'Air Quality Zone Code where the station/cabine is located',
    'European code to identify the station/cabine',
    'Longitude',
    'Latitude',
    'Address/Crossroads',
    'District code',
    'District name',
    'Neighbourhood code',
    'Name of the neighbourhood',
    'Type of station - classification 1',
    'Type of station - classification 2',
    'Pollutant code'
]

columns_from_2019_foward = {
    col: {
        'en': columns_from_2019_foward_en[i],
        'en_description': columns_from_2019_foward_en_description[i]
    } for i, col in enumerate(columns_from_2019_foward_es)
}

print(columns_from_2019_foward)

{'Estacio': {'en': 'station_number', 'en_description': 'Station number'}, 'nom_cabina': {'en': 'station_name', 'en_description': 'Name of the station'}, 'codi_dtes': {'en': 'station_code', 'en_description': 'Station code'}, 'zqa': {'en': 'aqzc', 'en_description': 'Air Quality Zone Code where the station/cabine is located'}, 'codi_eoi': {'en': 'eoi_code', 'en_description': 'European code to identify the station/cabine'}, 'Longitud': {'en': 'longitude', 'en_description': 'Longitude'}, 'Latitud': {'en': 'latitude', 'en_description': 'Latitude'}, 'ubicacio': {'en': 'location', 'en_description': 'Address/Crossroads'}, 'Codi_districte': {'en': 'district_code', 'en_description': 'District code'}, 'Nom_districte': {'en': 'district_name', 'en_description': 'District name'}, 'Codi_barri': {'en': 'neighborhood_code', 'en_description': 'Neighbourhood code'}, 'Nom_barri': {'en': 'neighborhood_name', 'en_description': 'Name of the neighbourhood'}, 'Clas_1': {'en': 'class_1', 'en_description': 'Type 

## EDA 2018

In [30]:
df_2018_stations = pd.read_csv(bronze_stations_folder_path + '/2018_stations.csv')

In [32]:
# 7 Rows and 3 Columns
df_2018_stations.shape

(7, 16)

In [35]:
# Display the first 5 rows of the DataFrame
df_2018_stations.head()

,nom_cabina,codi_dtes,zqa,codi_eoi,longitud,latitud,Ubicacio,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Ocupacio_sol,Emissions_Properes,Contaminant_1,Contaminant_2,Contaminant_3
0,Barcelona - Ciutadella,IL,1,8019050,2.1874,41.3864,Parc de la Ciutadella,1,Ciutat Vella,4,"Sant Pere, Santa Caterina i la Ribera",Urbana,Fons,NO2,O3,NaN
1,Barcelona - Eixample,IH,1,8019043,2.1538,41.3853,Av. Roma - c/ Comte Urgell,5,Eixample,9,la Nova Esquerra de l'Eixample,Urbana,Trànsit,NO2,O3,PM10
2,Barcelona - Gràcia,IJ,1,8019044,2.1534,41.3987,Plaça Gal·la Placídia (Via Augusta - Travesser...,6,Gracia,31,la Vila de Gracia,Urbana,Trànsit,NO2,O3,PM10
3,Barcelona - Palau Reial,IZ,1,8019057,2.1151,41.3875,c/ John Maynard Keynes - c/ de Jordi Girona,4,Les Corts,21,Pedralbes,Urbana,Fons,NO2,O3,PM10
4,Barcelona - Poblenou,I2,1,8019004,2.2045,41.4039,Plaça Josep Trueta (Pujades - Lope de Vega),10,Sant Marti,68,el Poblenou,Urbana,Fons,NO2,NaN,PM10


In [36]:
# Summary of the DataFrame including column names, their data types, 
# and their non-null values count
df_2018_stations.info()

# 'Codi_Contaminant' -> int code -> Contaminant code
# 'Desc_Contaminant' -> object/string (text) -> Contaminant description
# 'Unitats'          -> object/string (text) -> Contaminant units of measurement


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nom_cabina          7 non-null      object 
 1   codi_dtes           7 non-null      object 
 2   zqa                 7 non-null      int64  
 3   codi_eoi            7 non-null      int64  
 4   longitud            7 non-null      float64
 5   latitud             7 non-null      float64
 6   Ubicacio            7 non-null      object 
 7   Codi_Districte      7 non-null      int64  
 8   Nom_Districte       7 non-null      object 
 9   Codi_Barri          7 non-null      int64  
 10  Nom_Barri           7 non-null      object 
 11  Ocupacio_sol        7 non-null      object 
 12  Emissions_Properes  7 non-null      object 
 13  Contaminant_1       7 non-null      object 
 14  Contaminant_2       5 non-null      object 
 15  Contaminant_3       5 non-null      object 
dtypes: float64(2

In [ ]:
# Missing values for each column
missing_values = df_bronze_contaminants.isnull().sum()
missing_values

# Result: 2 NULL or NaN values in the 'Unitats' column
# Real result (by observation of dataset): 3 NULL or NaN values in the 'Unitats' column
# Rows 17, 18, 19

In [ ]:
# Missing values for each column
missing_values = df_bronze_contaminants.isna().sum()
missing_values

# Result: 2 NULL or NaN values in the 'Unitats' column
# Real result (by observation of dataset): 3 NULL or NaN values in the 'Unitats' column
# Rows 17, 18, 19

In [ ]:
# Get duplicated rows
duplicates = df_bronze_contaminants.duplicated().sum()
print("Duplicates: ", duplicates)

# No duplicated rows
# In reality we have multiple contaminants with 2 ids 
# (same 'Desc_Contaminant' but with an extra '*' char)

In [ ]:
# New DataFrame columns names
new_df_columns_names = ['code', 'description', 'unit']

# Change columns of DataFrame
df_bronze_contaminants.columns = new_df_columns_names


melhorar print de colunas em falta (diferença de colunas de um ano para o outro)

eda de dados por ano (começar por 2019) - usar antes pandas profilling
    fazer manual apenas 1 ano
IGNORAR 2018 (simplifica bastante) .. apenas fazer eda para perceber que pontos tem e que poluentes se focava

rever condecacao dos dados e começar a colocar código 
unificar dados e listar quais os poluentes por estacao e por ano
criar estrutura json com os columas unificadas(com metadata e tudo)
    criar um script para fazer load disso

usar os primeiros poluentes estudados como grau de importância 
saber que estacoes desapareceram ou apareceram ao longo dos anos
saber que poluenetes desapareceram ou apareceram ao longo dos anos (por estacao)
